In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

data = pd.read_csv("../data/processed/analysis_dataset.csv", index_col=0, parse_dates=True)
data.head()

,BAC,GS,JPM,SP500,DGS10,DGS2,VIX,ret_JPM,ret_BAC,ret_GS,ret_SP500,d_DGS10,d_DGS2,d_VIX
2015-01-05,13.573362,149.305878,44.636642,2020.579956,2.04,0.68,19.920000,-0.031537,-0.029481,-0.031720,-0.018447,-0.08,0.02,2.129999
2015-01-06,13.167254,146.285431,43.479256,2002.609985,1.97,0.65,21.120001,-0.026271,-0.030376,-0.020437,-0.008933,-0.07,-0.03,1.200001
2015-01-07,13.229733,148.465546,43.545605,2025.900024,1.96,0.62,19.309999,0.001525,0.004734,0.014793,0.011563,-0.01,-0.03,-1.810001
2015-01-08,13.503080,150.835876,44.518688,2062.139893,2.03,0.62,17.010000,0.022100,0.020451,0.015839,0.017730,0.07,0.00,-2.299999
2015-01-09,13.260975,148.521027,43.744637,2044.810059,1.98,0.59,17.549999,-0.017540,-0.018092,-0.015466,-0.008439,-0.05,-0.03,0.539999


In [3]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    return {
        "variable": name,
        "adf_stat": result[0],
        "p_value": result[1],
        "stationary": result[1] < 0.05
    }

variables_to_test = ["ret_JPM", "ret_BAC", "ret_GS", "ret_SP500", "d_DGS10", "d_DGS2", "d_VIX"]
adf_results = pd.DataFrame([adf_test(data[v], v) for v in variables_to_test])
adf_results

,variable,adf_stat,p_value,stationary
0,ret_JPM,-15.610212,1.783739e-28,True
1,ret_BAC,-17.248309,6.045938e-30,True
2,ret_GS,-17.103653,7.447393e-30,True
3,ret_SP500,-17.471795,4.538542e-30,True
4,d_DGS10,-40.533721,0.000000e+00,True
5,d_DGS2,-8.270828,4.798534e-13,True
6,d_VIX,-12.521618,2.541026e-23,True


In [6]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    return {
        "variable": name,
        "adf_stat": result[0],
        "p_value": result[1],
        "stationary": result[1] < 0.05
    }

variables_to_test = ["ret_JPM", "ret_BAC", "ret_GS", "ret_SP500", "d_DGS10", "d_DGS2", "d_VIX"]
adf_results = pd.DataFrame([adf_test(data[v], v) for v in variables_to_test])
adf_results

,variable,adf_stat,p_value,stationary
0,ret_JPM,-15.610212,1.783739e-28,True
1,ret_BAC,-17.248309,6.045938e-30,True
2,ret_GS,-17.103653,7.447393e-30,True
3,ret_SP500,-17.471795,4.538542e-30,True
4,d_DGS10,-40.533721,0.000000e+00,True
5,d_DGS2,-8.270828,4.798534e-13,True
6,d_VIX,-12.521618,2.541026e-23,True


In [9]:
from linearmodels.system import SUR

equations = {
    "JPM": {"dependent": data["ret_JPM"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX", "ret_SP500"]])},
    "BAC": {"dependent": data["ret_BAC"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX", "ret_SP500"]])},
    "GS": {"dependent": data["ret_GS"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX", "ret_SP500"]])},
}

sur_model = SUR(equations)
sur_results = sur_model.fit(cov_type="robust")
print(sur_results)

                           System GLS Estimation Summary                           
Estimator:                        GLS   Overall R-squared:                   0.5454
No. Equations.:                     3   McElroy's R-squared:                 0.3525
No. Observations:                2914   Judge's (OLS) R-squared:             0.5454
Date:                Sat, Sep 05 2026   Berndt's R-squared:                  0.6297
Time:                        16:26:48   Dhrymes's R-squared:                 0.5454
                                        Cov. Estimator:                      robust
                                        Num. Constraints:                      None
                  Equation: JPM, Dependent Variable: ret_JPM                  
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          0.0002     0.0002     0.8644     0.3874     -0.0002      0.0006
d_DGS10     

In [12]:
import numpy as np
from scipy import stats

idx_jpm = params.index.get_loc("JPM_d_DGS10")
idx_bac = params.index.get_loc("BAC_d_DGS10")
idx_gs = params.index.get_loc("GS_d_DGS10")

R = np.zeros((2, len(params)))
R[0, idx_jpm] = 1
R[0, idx_bac] = -1
R[1, idx_jpm] = 1
R[1, idx_gs] = -1

r_vec = R @ params.values
cov_matrix = cov.values
wald_stat = r_vec @ np.linalg.inv(R @ cov_matrix @ R.T) @ r_vec

p_value = 1 - stats.chi2.cdf(wald_stat, df=2)

print(f"Wald statistic: {wald_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"\nJoint null hypothesis: JPM_beta = BAC_beta = GS_beta")
print(f"Reject at 5%: {p_value < 0.05}")

Wald statistic: 63.4167
P-value: 0.0000

Joint null hypothesis: JPM_beta = BAC_beta = GS_beta
Reject at 5%: True


In [13]:
def chow_test(data, break_date, y_col, x_cols):
    before = data[data.index < break_date]
    after = data[data.index >= break_date]
    
    def rss(subset):
        y = subset[y_col]
        X = sm.add_constant(subset[x_cols])
        model = sm.OLS(y, X).fit()
        return model.ssr, len(subset), model.df_model + 1
    
    rss_pooled, n_pooled, k = rss(data)
    rss_before, n1, _ = rss(before)
    rss_after, n2, _ = rss(after)
    
    numerator = (rss_pooled - (rss_before + rss_after)) / k
    denominator = (rss_before + rss_after) / (n1 + n2 - 2*k)
    f_stat = numerator / denominator
    p_value = 1 - stats.f.cdf(f_stat, k, n1 + n2 - 2*k)
    
    return f_stat, p_value

f_stat, p_val = chow_test(data, "2022-03-01", "ret_JPM", ["d_DGS10", "d_VIX", "ret_SP500"])
print(f"Chow test F-statistic: {f_stat:.4f}")
print(f"P-value: {p_val:.4f}")

Chow test F-statistic: 24.0294
P-value: 0.0000


In [14]:
for bank, col in [("JPM", "ret_JPM"), ("BAC", "ret_BAC"), ("GS", "ret_GS")]:
    f_stat, p_val = chow_test(data, "2022-03-01", col, ["d_DGS10", "d_VIX", "ret_SP500"])
    print(f"{bank}: F-statistic = {f_stat:.4f}, p-value = {p_val:.4f}")

JPM: F-statistic = 24.0294, p-value = 0.0000
BAC: F-statistic = 56.4526, p-value = 0.0000
GS: F-statistic = 22.5153, p-value = 0.0000
